In [5]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

In [6]:
train_images_path = "archive/train-images.idx3-ubyte"
train_labels_path = "archive/train-labels.idx1-ubyte"

test_images_path = "archive/t10k-images.idx3-ubyte"
test_labels_path = "archive/t10k-labels.idx1-ubyte"

In [7]:
def read_images(file_path):
    with open(file_path, "rb") as file:
        header = np.frombuffer(file.read(16), dtype=">u4")
        magic_number, image_count, row_count, column_count = header

        pixel_data = file.read()
        pixels = np.frombuffer(pixel_data, dtype=np.uint8)

        images = pixels.reshape(image_count, row_count, column_count)
    
    return images, magic_number, image_count, row_count, column_count

In [8]:
def read_labels(file_path):
    with open(file_path, "rb") as file:
        header = np.frombuffer(file.read(8), dtype=">u4")
        magic_number, label_count = header

        label_data = file.read()
        labels = np.frombuffer(label_data, dtype=np.uint8)

    return labels, magic_number, label_count

In [9]:
train_images, train_image_magic_number, train_image_count ,train_image_row_count, train_image_column_count = read_images(train_images_path)
train_labels, train_label_magic_count, train_label_count = read_labels(train_labels_path)

test_images, test_image_magic_number, test_image_count, test_image_row_count, test_image_column_count = read_images(test_images_path)
test_labels, test_label_magic_count, test_label_count = read_labels(test_labels_path)

print("Train images: ", train_images.shape)
print("Train labels: ", train_labels.shape)
print("test_images: ", test_images.shape)
print("Test labels: ", test_labels.shape)

print("Train image magic number: ", train_image_magic_number)
print("Train label magic number: ", train_label_magic_count)
print("Test image magic number: ", test_image_magic_number)
print("Test label magic number: ", test_label_magic_count)

Train images:  (60000, 28, 28)
Train labels:  (60000,)
test_images:  (10000, 28, 28)
Test labels:  (10000,)
Train image magic number:  2051
Train label magic number:  2049
Test image magic number:  2051
Test label magic number:  2049


In [10]:
train_vectors = train_images.reshape(train_images.shape[0], train_images.shape[1] * train_images.shape[2])
test_vectors = test_images.reshape(test_images.shape[0], test_images.shape[1] * test_images.shape[2])

normalized_train_vectors = train_vectors.astype(np.float32) / 255.0
normalized_test_vectors = test_vectors.astype(np.float32) / 255.0

In [11]:
random_seed = 42
validation_per_class = 1000

random_generator = np.random.default_rng(random_seed)

validation_indices_list = []
training_indices_list = []

for class_label in range(10):
    class_indices = np.where(train_labels == class_label)[0]
    stuffed_class_indices = random_generator.permutation(class_indices)

    validation_indices_list.append(stuffed_class_indices[:validation_per_class])
    training_indices_list.append(stuffed_class_indices[validation_per_class:])

validation_indices = np.concatenate(validation_indices_list)
training_indices = np.concatenate(training_indices_list)

validation_indices = random_generator.permutation(validation_indices)
training_indices = random_generator.permutation(training_indices)

model_train_vectors = normalized_train_vectors[training_indices]
model_train_labels = train_labels[training_indices]

validation_vectors = normalized_train_vectors[validation_indices]
validation_labels = train_labels[validation_indices]

test_vectors_normalized = normalized_test_vectors

In [12]:
def calculate_accuracy(true_labels, predicted_labels):
    return np.sum(true_labels == predicted_labels) / true_labels.shape[0]

def calculate_confusion_matrix(true_labels, predicted_labels, class_count=10):
    matrix = np.zeros((class_count, class_count), dtype=int)

    for true_label, predicted_label in zip(true_labels, predicted_labels):
        matrix[true_label, predicted_label] += 1

    return matrix

In [13]:
validation_results = []
test_results = []

In [20]:
def run_decision_tree_validation_search(max_depth_values, min_samples_split_values, stage_name):
    results = []
    criteria = ["gini", "entropy"]

    total_experiment_count = len(criteria) * len(max_depth_values) * len(min_samples_split_values)
    experiment_no = 1

    for criterion in criteria:
        for max_depth in max_depth_values:
            for min_samples_split in min_samples_split_values:

                model = DecisionTreeClassifier(
                    criterion=criterion,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    random_state=42
                )

                model.fit(model_train_vectors, model_train_labels)
                validation_accuracy = model.score(validation_vectors, validation_labels)

                results.append({
                    "model": "Decision Tree",
                    "feature_type": "Raw Vectors",
                    "search_stage": stage_name,
                    "criterion": criterion,
                    "max_depth": max_depth,
                    "min_samples_split": min_samples_split,
                    "validation_accuracy": validation_accuracy
                })

                if experiment_no % 50 == 0:
                    print("Completed", experiment_no, "of", total_experiment_count)

                experiment_no += 1

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="validation_accuracy", ascending=False)

    return results_df

In [21]:
coarse_max_depth_values = [1, 2, 4, 8, 16, 32, 64] # başlangıçta 128 de eklemiştim ama çok uzun sürüyor
coarse_min_samples_split_values = [2, 10, 50, 100, 250, 500, 1000]

decision_tree_raw_coarse_result_df = run_decision_tree_validation_search(
    max_depth_values=coarse_max_depth_values,
    min_samples_split_values=coarse_min_samples_split_values,
    stage_name="coarse_search"
)

decision_tree_raw_coarse_result_df.head(10)

Completed 50 of 98


,model,feature_type,search_stage,criterion,max_depth,min_samples_split,validation_accuracy
91,Decision Tree,Raw Vectors,coarse_search,entropy,64,2,0.8739
84,Decision Tree,Raw Vectors,coarse_search,entropy,32,2,0.8739
77,Decision Tree,Raw Vectors,coarse_search,entropy,16,2,0.8726
92,Decision Tree,Raw Vectors,coarse_search,entropy,64,10,0.8703
85,Decision Tree,Raw Vectors,coarse_search,entropy,32,10,0.8703
78,Decision Tree,Raw Vectors,coarse_search,entropy,16,10,0.8699
28,Decision Tree,Raw Vectors,coarse_search,gini,16,2,0.8674
29,Decision Tree,Raw Vectors,coarse_search,gini,16,10,0.8657
36,Decision Tree,Raw Vectors,coarse_search,gini,32,10,0.8642
35,Decision Tree,Raw Vectors,coarse_search,gini,32,2,0.8640


In [22]:
best_coarse_decision_tree_raw = decision_tree_raw_coarse_result_df.iloc[0]

best_coarse_max_depth = int(best_coarse_decision_tree_raw["max_depth"])
best_coarse_min_samples_split = int(best_coarse_decision_tree_raw["min_samples_split"])

print(
    best_coarse_decision_tree_raw["criterion"],
    best_coarse_max_depth,
    best_coarse_min_samples_split,
    best_coarse_decision_tree_raw["validation_accuracy"]
)

# max_depth için refined search aralığı
best_depth_index = coarse_max_depth_values.index(best_coarse_max_depth)

if best_depth_index == 0:
    lower_depth = 1
    upper_depth = coarse_max_depth_values[best_depth_index + 1]
elif best_depth_index == len(coarse_max_depth_values) - 1:
    lower_depth = coarse_max_depth_values[best_depth_index - 1]
    upper_depth = best_coarse_max_depth * 2
else:
    lower_depth = coarse_max_depth_values[best_depth_index - 1]
    upper_depth = coarse_max_depth_values[best_depth_index + 1]

refined_max_depth_values = sorted(set(
    int(x) for x in np.round(np.linspace(lower_depth, upper_depth, 7))
))

if best_coarse_max_depth not in refined_max_depth_values:
    refined_max_depth_values.append(best_coarse_max_depth)
    refined_max_depth_values = sorted(refined_max_depth_values)


# min_samples_split için refined search aralığı
best_min_index = coarse_min_samples_split_values.index(best_coarse_min_samples_split)

if best_min_index == 0:
    lower_min_samples_split = 2
    upper_min_samples_split = coarse_min_samples_split_values[best_min_index + 1]
elif best_min_index == len(coarse_min_samples_split_values) - 1:
    lower_min_samples_split = coarse_min_samples_split_values[best_min_index - 1]
    upper_min_samples_split = best_coarse_min_samples_split + 500
else:
    lower_min_samples_split = coarse_min_samples_split_values[best_min_index - 1]
    upper_min_samples_split = coarse_min_samples_split_values[best_min_index + 1]

refined_min_samples_split_values = sorted(set(
    max(2, int(x)) for x in np.round(
        np.linspace(lower_min_samples_split, upper_min_samples_split, 7)
    )
))

if best_coarse_min_samples_split not in refined_min_samples_split_values:
    refined_min_samples_split_values.append(best_coarse_min_samples_split)
    refined_min_samples_split_values = sorted(refined_min_samples_split_values)

print("Refined max_depth values:", refined_max_depth_values)
print("Refined min_samples_split values:", refined_min_samples_split_values)

decision_tree_raw_refined_result_df = run_decision_tree_validation_search(
    max_depth_values=refined_max_depth_values,
    min_samples_split_values=refined_min_samples_split_values,
    stage_name="refined_search"
)

decision_tree_raw_results_df = pd.concat(
    [decision_tree_raw_coarse_result_df, decision_tree_raw_refined_result_df],
    ignore_index=True
)

decision_tree_raw_results_df = decision_tree_raw_results_df.sort_values(
    by="validation_accuracy",
    ascending=False
)

best_decision_tree_raw = decision_tree_raw_results_df.iloc[0]

best_dt_raw_criterion = best_decision_tree_raw["criterion"]
best_dt_raw_max_depth = int(best_decision_tree_raw["max_depth"])
best_dt_raw_min_samples_split = int(best_decision_tree_raw["min_samples_split"])

print(
    best_dt_raw_criterion,
    best_dt_raw_max_depth,
    best_dt_raw_min_samples_split,
    best_decision_tree_raw["validation_accuracy"]
)

decision_tree_raw_results_df.head(10)

entropy 64 2 0.8739
Refined max_depth values: [32, 48, 64, 80, 96, 112, 128]
Refined min_samples_split values: [2, 3, 5, 6, 7, 9, 10]
Completed 50 of 98
entropy 64 2 0.8739


,model,feature_type,search_stage,criterion,max_depth,min_samples_split,validation_accuracy
0,Decision Tree,Raw Vectors,coarse_search,entropy,64,2,0.8739
1,Decision Tree,Raw Vectors,coarse_search,entropy,32,2,0.8739
99,Decision Tree,Raw Vectors,refined_search,entropy,32,2,0.8739
98,Decision Tree,Raw Vectors,refined_search,entropy,48,2,0.8739
104,Decision Tree,Raw Vectors,refined_search,entropy,64,2,0.8739
100,Decision Tree,Raw Vectors,refined_search,entropy,80,2,0.8739
102,Decision Tree,Raw Vectors,refined_search,entropy,128,2,0.8739
103,Decision Tree,Raw Vectors,refined_search,entropy,112,2,0.8739
101,Decision Tree,Raw Vectors,refined_search,entropy,96,2,0.8739
105,Decision Tree,Raw Vectors,refined_search,entropy,48,3,0.8727


In [23]:
validation_results.append({
    "model": "Decision Tree",
    "feature_type": "Raw Vectors",
    "best_parameters": {
        "criterion": best_dt_raw_criterion,
        "max_depth": best_dt_raw_max_depth,
        "min_samples_split": best_dt_raw_min_samples_split
    },
    "validation_accuracy": best_decision_tree_raw["validation_accuracy"]
})

pd.DataFrame(validation_results)

,model,feature_type,best_parameters,validation_accuracy
0,Decision Tree,Raw Vectors,"{'criterion': 'entropy', 'max_depth': 64, 'min...",0.8739


In [24]:
def run_random_forest_validation_search(n_estimators_values, max_depth_values, max_features_values, stage_name):
    results = []

    for n_estimators in n_estimators_values:
        for max_depth in max_depth_values:
            for max_features in max_features_values:

                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    max_features=max_features,
                    random_state=42,
                    n_jobs=-1
                )

                model.fit(model_train_vectors, model_train_labels)
                validation_predictions = model.predict(validation_vectors)
                validation_accuracy = calculate_accuracy(validation_labels, validation_predictions)

                results.append({
                    "model": "Random Forest",
                    "feature_type": "Raw Vectors",
                    "search_stage": stage_name,
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "max_features": max_features,
                    "validation_accuracy": validation_accuracy
                })

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="validation_accuracy", ascending=False)

    return results_df

In [25]:
rf_coarse_n_estimators_values = [50, 100, 200]
rf_coarse_max_depth_values = [4, 8, 16, 32, 64] # 128 çıkarıldı, çok uzun sürüyor
rf_coarse_max_features_values = ["sqrt", "log2"]

random_forest_raw_coarse_result_df = run_random_forest_validation_search(
    n_estimators_values=rf_coarse_n_estimators_values,
    max_depth_values=rf_coarse_max_depth_values,
    max_features_values=rf_coarse_max_features_values,
    stage_name = "coarse_search"
)

random_forest_raw_coarse_result_df.head(10)

,model,feature_type,search_stage,n_estimators,max_depth,max_features,validation_accuracy
26,Random Forest,Raw Vectors,coarse_search,200,32,sqrt,0.9686
28,Random Forest,Raw Vectors,coarse_search,200,64,sqrt,0.9680
18,Random Forest,Raw Vectors,coarse_search,100,64,sqrt,0.9676
16,Random Forest,Raw Vectors,coarse_search,100,32,sqrt,0.9672
29,Random Forest,Raw Vectors,coarse_search,200,64,log2,0.9665
27,Random Forest,Raw Vectors,coarse_search,200,32,log2,0.9660
24,Random Forest,Raw Vectors,coarse_search,200,16,sqrt,0.9658
17,Random Forest,Raw Vectors,coarse_search,100,32,log2,0.9657
19,Random Forest,Raw Vectors,coarse_search,100,64,log2,0.9650
6,Random Forest,Raw Vectors,coarse_search,50,32,sqrt,0.9647


In [26]:
best_coarse_random_forest_raw = random_forest_raw_coarse_result_df.iloc[0]

best_coarse_n_estimators = int(best_coarse_random_forest_raw["n_estimators"])
best_coarse_rf_max_depth = best_coarse_random_forest_raw["max_depth"]
best_coarse_max_features = best_coarse_random_forest_raw["max_features"]

print(best_coarse_n_estimators, best_coarse_rf_max_depth, best_coarse_max_features, best_coarse_random_forest_raw["validation_accuracy"])

# n_estimators çevresi
best_n_index = rf_coarse_n_estimators_values.index(best_coarse_n_estimators)

if best_n_index == 0:
    refined_n_estimators_values = list(range(25, rf_coarse_n_estimators_values[best_n_index + 1] + 1, 25))
elif best_n_index == len(rf_coarse_n_estimators_values) - 1:
    refined_n_estimators_values = list(range(rf_coarse_n_estimators_values[best_n_index - 1], best_coarse_n_estimators + 101, 50))
else:
    refined_n_estimators_values = list(range(rf_coarse_n_estimators_values[best_n_index - 1], rf_coarse_n_estimators_values[best_n_index + 1] + 1, 25))


# max_depth çevresi
best_coarse_rf_max_depth = int(best_coarse_rf_max_depth)
best_depth_index = rf_coarse_max_depth_values.index(best_coarse_rf_max_depth)

if best_depth_index == 0:
    refined_rf_max_depth_values = list(range(2, rf_coarse_max_depth_values[best_depth_index + 1] + 1, 2))
elif best_depth_index == len(rf_coarse_max_depth_values) -1:
    refined_rf_max_depth_values = list(range(rf_coarse_max_depth_values[best_depth_index - 1], best_coarse_rf_max_depth + 65, 16))
else:
    refined_rf_max_depth_values = list(range(rf_coarse_max_depth_values[best_depth_index - 1], rf_coarse_max_depth_values[best_depth_index + 1] + 1, 4))

random_forest_raw_refined_result_df = run_random_forest_validation_search(
    n_estimators_values=refined_n_estimators_values,
    max_depth_values=refined_rf_max_depth_values,
    max_features_values=["sqrt", "log2"],
    stage_name="refined_search"
)

200 32 sqrt 0.9686


In [27]:
random_forest_raw_results_df = pd.concat([random_forest_raw_coarse_result_df, random_forest_raw_refined_result_df], ignore_index=True)
random_forest_raw_results_df = random_forest_raw_results_df.sort_values(by="validation_accuracy", ascending=False)

best_random_forest_raw = random_forest_raw_results_df.iloc[0]

best_rf_raw_n_estimators = int(best_random_forest_raw["n_estimators"])
best_rf_raw_max_depth = int(best_random_forest_raw["max_depth"])
best_rf_raw_max_features = best_random_forest_raw["max_features"]

random_forest_raw_results_df.head(10)

,model,feature_type,search_stage,n_estimators,max_depth,max_features,validation_accuracy
30,Random Forest,Raw Vectors,refined_search,300,24,sqrt,0.9697
31,Random Forest,Raw Vectors,refined_search,300,32,sqrt,0.9694
32,Random Forest,Raw Vectors,refined_search,250,44,sqrt,0.9691
45,Random Forest,Raw Vectors,refined_search,300,40,sqrt,0.9690
43,Random Forest,Raw Vectors,refined_search,250,48,sqrt,0.9690
42,Random Forest,Raw Vectors,refined_search,300,64,sqrt,0.9690
41,Random Forest,Raw Vectors,refined_search,300,56,sqrt,0.9690
44,Random Forest,Raw Vectors,refined_search,300,48,sqrt,0.9690
39,Random Forest,Raw Vectors,refined_search,250,60,sqrt,0.9690
38,Random Forest,Raw Vectors,refined_search,250,56,sqrt,0.9690


In [28]:
validation_results.append({
    "model": "Random Forest",
    "feature_type": "Raw Vectors",
    "best_parameters": {
        "n_estimators": best_rf_raw_n_estimators,
        "max_depth": best_rf_raw_max_depth,
        "max_features": best_rf_raw_max_features
    },
    "validation_accuracy": best_random_forest_raw["validation_accuracy"]
})

pd.DataFrame(validation_results)

,model,feature_type,best_parameters,validation_accuracy
0,Decision Tree,Raw Vectors,"{'criterion': 'entropy', 'max_depth': 64, 'min...",0.8739
1,Random Forest,Raw Vectors,"{'n_estimators': 300, 'max_depth': 24, 'max_fe...",0.9697


In [29]:
gb_fixed_max_iter = 50

def run_gradient_boosting_validation_search(learning_rate_values, max_depth_values, stage_name):
    results = []

    for learning_rate in learning_rate_values:
        for max_depth in max_depth_values:
            model = HistGradientBoostingClassifier(
                learning_rate=learning_rate,
                max_depth=max_depth,
                max_iter=gb_fixed_max_iter,
                random_state=42,
                early_stopping=True,
                n_iter_no_change=5,
                validation_fraction=0.1,
                max_bins=32
            )

            model.fit(model_train_vectors, model_train_labels)
            predictions = model.predict(validation_vectors)
            accuracy = calculate_accuracy(validation_labels, predictions)

            results.append({
                "model": "Gradient Boosting",
                "feature_type": "Raw Vectors",
                "search_stage": stage_name,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "max_iter": gb_fixed_max_iter,
                "validation_accuracy": accuracy
            })

    return pd.DataFrame(results).sort_values(by="validation_accuracy", ascending=False)

In [30]:
gb_coarse_learning_rate_values = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
gb_coarse_max_depth_values = [1, 2, 4, 8]

gradient_boosting_raw_coarse_result_df = run_gradient_boosting_validation_search(
    learning_rate_values=gb_coarse_learning_rate_values,
    max_depth_values=gb_coarse_max_depth_values,
    stage_name="coarse_search"
)

gradient_boosting_raw_coarse_result_df.head(10)

,model,feature_type,search_stage,learning_rate,max_depth,max_iter,validation_accuracy
19,Gradient Boosting,Raw Vectors,coarse_search,0.100,8,50,0.9669
23,Gradient Boosting,Raw Vectors,coarse_search,0.300,8,50,0.9521
18,Gradient Boosting,Raw Vectors,coarse_search,0.100,4,50,0.9505
22,Gradient Boosting,Raw Vectors,coarse_search,0.300,4,50,0.9475
15,Gradient Boosting,Raw Vectors,coarse_search,0.030,8,50,0.9431
21,Gradient Boosting,Raw Vectors,coarse_search,0.300,2,50,0.9257
11,Gradient Boosting,Raw Vectors,coarse_search,0.010,8,50,0.9182
14,Gradient Boosting,Raw Vectors,coarse_search,0.030,4,50,0.9116
17,Gradient Boosting,Raw Vectors,coarse_search,0.100,2,50,0.8971
7,Gradient Boosting,Raw Vectors,coarse_search,0.003,8,50,0.8866


In [31]:
best_coarse_gradient_boosting_raw = gradient_boosting_raw_coarse_result_df.iloc[0]

best_coarse_learning_rate = float(best_coarse_gradient_boosting_raw["learning_rate"])
best_coarse_gb_max_depth = int(best_coarse_gradient_boosting_raw["max_depth"])

best_lr_index = gb_coarse_learning_rate_values.index(best_coarse_learning_rate)
best_depth_index = gb_coarse_max_depth_values.index(best_coarse_gb_max_depth)

if best_lr_index == 0:
    lower_lr = gb_coarse_learning_rate_values[0]
    upper_lr = gb_coarse_learning_rate_values[1]
elif best_lr_index == len(gb_coarse_learning_rate_values) - 1:
    lower_lr = gb_coarse_learning_rate_values[-2]
    upper_lr = gb_coarse_learning_rate_values[-1] * 1.5
else:
    lower_lr = gb_coarse_learning_rate_values[best_lr_index - 1]
    upper_lr = gb_coarse_learning_rate_values[best_lr_index + 1]

refined_learning_rate_values = list(np.round(np.linspace(lower_lr, upper_lr, 7), 4))

if best_depth_index == 0:
    refined_gb_max_depth_values = list(range(1, gb_coarse_max_depth_values[best_depth_index + 1] + 1))
elif best_depth_index == len(gb_coarse_max_depth_values) - 1:
    refined_gb_max_depth_values = list(range(
        gb_coarse_max_depth_values[best_depth_index - 1],
        best_coarse_gb_max_depth * 2 + 1,
        2
    ))
else:
    refined_gb_max_depth_values = list(range(gb_coarse_max_depth_values[best_depth_index - 1], gb_coarse_max_depth_values[best_depth_index + 1] + 1))

gradient_boosting_raw_refined_result_df = run_gradient_boosting_validation_search(
    learning_rate_values=refined_learning_rate_values,
    max_depth_values=refined_gb_max_depth_values,
    stage_name="refined_search"
)

In [32]:
gradient_boosting_raw_results_df = pd.concat([gradient_boosting_raw_coarse_result_df, gradient_boosting_raw_refined_result_df], ignore_index=True)
gradient_boosting_raw_results_df = gradient_boosting_raw_results_df.sort_values(by="validation_accuracy", ascending=False)

best_gradient_boosting_raw = gradient_boosting_raw_results_df.iloc[0]

best_gb_raw_learning_rate = float(best_gradient_boosting_raw["learning_rate"])
best_gb_raw_max_depth = int(best_gradient_boosting_raw["max_depth"])

gradient_boosting_raw_results_df.head(10)

,model,feature_type,search_stage,learning_rate,max_depth,max_iter,validation_accuracy
24,Gradient Boosting,Raw Vectors,refined_search,0.210,10,50,0.9756
25,Gradient Boosting,Raw Vectors,refined_search,0.255,14,50,0.9752
26,Gradient Boosting,Raw Vectors,refined_search,0.255,16,50,0.9744
27,Gradient Boosting,Raw Vectors,refined_search,0.210,8,50,0.9738
28,Gradient Boosting,Raw Vectors,refined_search,0.210,12,50,0.9735
29,Gradient Boosting,Raw Vectors,refined_search,0.210,14,50,0.9732
30,Gradient Boosting,Raw Vectors,refined_search,0.210,6,50,0.9732
31,Gradient Boosting,Raw Vectors,refined_search,0.210,16,50,0.9731
32,Gradient Boosting,Raw Vectors,refined_search,0.165,10,50,0.9729
33,Gradient Boosting,Raw Vectors,refined_search,0.165,14,50,0.9728


In [33]:
validation_results.append({
    "model": "Gradient Boosting",
    "feature_type": "Raw Vectors",
    "best_parameters": {
        "learning_rate": best_gb_raw_learning_rate,
        "max_depth": best_gb_raw_max_depth,
        "max_iter": gb_fixed_max_iter
    },
    "validation_accuracy": best_gradient_boosting_raw["validation_accuracy"]
})

In [34]:
pca_max_component_count = 256 # başlangıçta 512'ydi çok uzun sürdüğü için azalttım

pca_model = PCA(n_components=pca_max_component_count, svd_solver="randomized", random_state=42)

pca_model_train_all = pca_model.fit_transform(model_train_vectors)
pca_validation_all = pca_model.transform(validation_vectors)
pca_test_all = pca_model.transform(test_vectors_normalized)

pca_preserved_variance = np.cumsum(pca_model.explained_variance_ratio_)

In [35]:
pca_coarse_component_values = [16, 32, 64, 128, 192, 256] # 512 çıkarıldı, çok maliyetli

In [36]:
def create_refined_pca_values(best_component, coarse_values):
    best_index = coarse_values.index(best_component)

    if best_index == 0:
        lower_component = coarse_values[0]
        upper_componenet = coarse_values[best_index + 1]
    elif best_index == len(coarse_values) - 1:
        lower_component = coarse_values[best_index - 1]
        upper_componenet = coarse_values[-1]
    else:
        lower_component = coarse_values[best_index - 1]
        upper_componenet = coarse_values[best_index + 1]
    
    if upper_componenet <= 128:
        component_step = 8
    else:
        component_step = 16
    
    refined_values = list(range(lower_component, upper_componenet + 1, component_step))

    if best_component not in refined_values:
        refined_values.append(best_component)
        refined_values = sorted(refined_values)
    
    return refined_values

In [44]:
def run_decision_tree_pca_search(
    pca_componenet_values,
    max_depth_values,
    min_sample_split_values,
    stage_name,
    criteria=None
):
    results = []

    if criteria is None:
        criteria = ["gini", "entropy"]

    for component_count in pca_componenet_values:
        current_train_vectors = np.ascontiguousarray(
            pca_model_train_all[:, :component_count]
        )
        current_validation_vectors = np.ascontiguousarray(
            pca_validation_all[:, :component_count]
        )
        preserved_variance = pca_preserved_variance[component_count - 1]

        for criterion in criteria:
            for max_depth in max_depth_values:
                for min_samples_split in min_sample_split_values:

                    model = DecisionTreeClassifier(
                        criterion=criterion,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        random_state=42
                    )

                    model.fit(current_train_vectors, model_train_labels)
                    accuracy = model.score(current_validation_vectors, validation_labels)

                    results.append({
                        "model": "Decision Tree",
                        "feature_type": "PCA Vectors",
                        "search_stage": stage_name,
                        "pca_components": component_count,
                        "preserved_variance": preserved_variance,
                        "criterion": criterion,
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "validation_accuracy": accuracy
                    })

    return pd.DataFrame(results).sort_values(
        by="validation_accuracy",
        ascending=False
    )

In [45]:
pca_coarse_component_values = [20, 100, 200]

dt_pca_coarse_max_depth_values = [8, 32, 64]
dt_pca_coarse_samples_split_values = [2, 500]

decision_tree_pca_coarse_result_df = run_decision_tree_pca_search(
    pca_componenet_values=pca_coarse_component_values,
    max_depth_values=dt_pca_coarse_max_depth_values,
    min_sample_split_values=dt_pca_coarse_samples_split_values,
    stage_name="coarse_search"
)

decision_tree_pca_coarse_result_df.head(10)

,model,feature_type,search_stage,pca_components,preserved_variance,criterion,max_depth,min_samples_split,validation_accuracy
8,Decision Tree,PCA Vectors,coarse_search,20,0.644021,entropy,32,2,0.8501
10,Decision Tree,PCA Vectors,coarse_search,20,0.644021,entropy,64,2,0.8501
4,Decision Tree,PCA Vectors,coarse_search,20,0.644021,gini,64,2,0.8408
2,Decision Tree,PCA Vectors,coarse_search,20,0.644021,gini,32,2,0.8398
22,Decision Tree,PCA Vectors,coarse_search,100,0.914768,entropy,64,2,0.8363
20,Decision Tree,PCA Vectors,coarse_search,100,0.914768,entropy,32,2,0.8363
14,Decision Tree,PCA Vectors,coarse_search,100,0.914768,gini,32,2,0.8306
34,Decision Tree,PCA Vectors,coarse_search,200,0.966518,entropy,64,2,0.8284
32,Decision Tree,PCA Vectors,coarse_search,200,0.966518,entropy,32,2,0.8284
16,Decision Tree,PCA Vectors,coarse_search,100,0.914768,gini,64,2,0.8283


In [46]:
best_dt_pca_coarse = decision_tree_pca_coarse_result_df.iloc[0]

best_dta_pca_component = int(best_dt_pca_coarse["pca_components"])
best_dt_pca_max_depth = int(best_dt_pca_coarse["max_depth"])
best_dt_pca_min_samples_split = int(best_dt_pca_coarse["min_samples_split"])

print(
    best_dta_pca_component,
    best_dt_pca_max_depth,
    best_dt_pca_min_samples_split,
    best_dt_pca_coarse["validation_accuracy"]
)

refined_pca_component_values = create_refined_pca_values(
    best_dta_pca_component,
    pca_coarse_component_values
)

# Çok fazla PCA değeri üretirse 5 değere indiriyoruz
if len(refined_pca_component_values) > 5:
    refined_pca_component_values = sorted(set(
        int(x) for x in np.round(
            np.linspace(
                min(refined_pca_component_values),
                max(refined_pca_component_values),
                5
            )
        )
    ))

if best_dta_pca_component not in refined_pca_component_values:
    refined_pca_component_values.append(best_dta_pca_component)
    refined_pca_component_values = sorted(refined_pca_component_values)


# Decision Tree PCA refined search değerleri
refined_dt_pca_max_depth_values = sorted(set([
    max(1, best_dt_pca_max_depth // 2),
    best_dt_pca_max_depth,
    best_dt_pca_max_depth + 32
]))

refined_dt_pca_min_samples_split_values = sorted(set([
    max(2, best_dt_pca_min_samples_split),
    max(5, best_dt_pca_min_samples_split + 3),
    max(10, best_dt_pca_min_samples_split + 8)
]))

print("Refined PCA component values:", refined_pca_component_values)
print("Refined DT PCA max_depth values:", refined_dt_pca_max_depth_values)
print("Refined DT PCA min_samples_split values:", refined_dt_pca_min_samples_split_values)

decision_tree_pca_refined_result_df = run_decision_tree_pca_search(
    pca_componenet_values=refined_pca_component_values,
    max_depth_values=refined_dt_pca_max_depth_values,
    min_sample_split_values=refined_dt_pca_min_samples_split_values,
    stage_name="refined_search",
    criteria=[best_dt_pca_coarse["criterion"]]
)

20 32 2 0.8501
Refined PCA component values: [20, 40, 60, 80, 100]
Refined DT PCA max_depth values: [16, 32, 64]
Refined DT PCA min_samples_split values: [2, 5, 10]


In [47]:
decision_tree_pca_results_df = pd.concat([decision_tree_pca_coarse_result_df, decision_tree_pca_refined_result_df], ignore_index=True)
decision_tree_pca_results_df = decision_tree_pca_results_df.sort_values(by="validation_accuracy", ascending=False)

best_decision_tree_pca = decision_tree_pca_results_df.iloc[0]

best_dt_pca_components = int(best_decision_tree_pca["pca_components"])
best_dt_pca_criterion = best_decision_tree_pca["criterion"]
best_dt_pca_max_depth = int(best_decision_tree_pca["max_depth"])
best_dt_pca_min_samples_split = int(best_decision_tree_pca["min_samples_split"])

decision_tree_pca_results_df.head(10)

,model,feature_type,search_stage,pca_components,preserved_variance,criterion,max_depth,min_samples_split,validation_accuracy
36,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,16,2,0.8512
1,Decision Tree,PCA Vectors,coarse_search,20,0.644021,entropy,64,2,0.8501
38,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,64,2,0.8501
37,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,32,2,0.8501
0,Decision Tree,PCA Vectors,coarse_search,20,0.644021,entropy,32,2,0.8501
39,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,64,5,0.8489
40,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,32,5,0.8489
41,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,16,10,0.8485
42,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,16,5,0.8476
43,Decision Tree,PCA Vectors,refined_search,20,0.644021,entropy,32,10,0.8468


In [48]:
validation_results.append({
    "model": "Decision Tree",
    "feature_type": "PCA Vectors",
    "best_parameters": {
        "pca_components": best_dt_pca_components,
        "criterion": best_dt_pca_criterion,
        "max_depth": best_dt_pca_max_depth,
        "min_samples_split": best_dt_pca_min_samples_split
    },
    "validation_accuracy": best_decision_tree_pca["validation_accuracy"]
})

pd.DataFrame(validation_results)

,model,feature_type,best_parameters,validation_accuracy
0,Decision Tree,Raw Vectors,"{'criterion': 'entropy', 'max_depth': 64, 'min...",0.8739
1,Random Forest,Raw Vectors,"{'n_estimators': 300, 'max_depth': 24, 'max_fe...",0.9697
2,Gradient Boosting,Raw Vectors,"{'learning_rate': 0.21, 'max_depth': 10, 'max_...",0.9756
3,Decision Tree,PCA Vectors,"{'pca_components': 20, 'criterion': 'entropy',...",0.8512


In [52]:
def run_random_forest_pca_search(pca_component_values, n_estimators_values, max_depth_values, max_features_values, stage_name):
    results = []

    for component_count in pca_component_values:
        current_train_vectors = pca_model_train_all[:, : component_count]
        current_validation_vectors = pca_validation_all[: , : component_count]
        preserved_variance = pca_preserved_variance[component_count - 1]

        for n_estimators in n_estimators_values:
            for max_depth in max_depth_values:
                for max_features in max_features_values:

                    model = RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        max_features=max_features,
                        random_state=42,
                        n_jobs=-1
                    )
                    
                    model.fit(current_train_vectors, model_train_labels)
                    predictions = model.predict(current_validation_vectors)
                    accuracy = calculate_accuracy(validation_labels, predictions)

                    results.append({
                        "model": "Random Forest",
                        "feature_type": "PCA Vectors",
                        "search_stage": stage_name,
                        "pca_components": component_count,
                        "preserved_variance": preserved_variance,
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "max_features": max_features,
                        "validation_accuracy": accuracy
                    })

    return pd.DataFrame(results).sort_values(by="validation_accuracy", ascending=False)

In [53]:
rf_pca_coarse_n_estimators_values = [50, 100, 200]
rf_pca_coarse_max_depth_values = [8, 16, 32, 64]
rf_pca_coarse_max_features_values = ["sqrt", "log2"]

random_forest_pca_coarse_result_df = run_random_forest_pca_search(
    pca_component_values=pca_coarse_component_values,
    n_estimators_values=rf_pca_coarse_n_estimators_values,
    max_depth_values=rf_pca_coarse_max_depth_values,
    max_features_values=rf_pca_coarse_max_features_values,
    stage_name="coarse_search"
)

random_forest_pca_coarse_result_df.head(10)

,model,feature_type,search_stage,pca_components,preserved_variance,n_estimators,max_depth,max_features,validation_accuracy
47,Random Forest,PCA Vectors,coarse_search,100,0.914768,200,64,log2,0.9505
45,Random Forest,PCA Vectors,coarse_search,100,0.914768,200,32,log2,0.9499
46,Random Forest,PCA Vectors,coarse_search,100,0.914768,200,64,sqrt,0.9487
44,Random Forest,PCA Vectors,coarse_search,100,0.914768,200,32,sqrt,0.9483
39,Random Forest,PCA Vectors,coarse_search,100,0.914768,100,64,log2,0.9478
37,Random Forest,PCA Vectors,coarse_search,100,0.914768,100,32,log2,0.9459
38,Random Forest,PCA Vectors,coarse_search,100,0.914768,100,64,sqrt,0.9458
20,Random Forest,PCA Vectors,coarse_search,20,0.644021,200,32,sqrt,0.9445
21,Random Forest,PCA Vectors,coarse_search,20,0.644021,200,32,log2,0.9445
68,Random Forest,PCA Vectors,coarse_search,200,0.966518,200,32,sqrt,0.9444


In [55]:
best_rf_pca_coarse = random_forest_pca_coarse_result_df.iloc[0]

best_rf_pca_component = int(best_rf_pca_coarse["pca_components"])
best_rf_pca_n_estimators = int(best_rf_pca_coarse["n_estimators"])
best_rf_pca_max_depth = int(best_rf_pca_coarse["max_depth"])
best_rf_pca_max_features = best_rf_pca_coarse["max_features"]

refined_rf_pca_component_values = create_refined_pca_values(best_rf_pca_component, pca_coarse_component_values)
best_n_index = rf_pca_coarse_n_estimators_values.index(best_rf_pca_n_estimators)

if best_n_index == 0:
    refined_rf_pca_n_estimators_values = list(range(50, rf_pca_coarse_n_estimators_values[best_n_index + 1] + 1, 50))
elif best_n_index == len(rf_pca_coarse_n_estimators_values) - 1:
    refined_rf_pca_n_estimators_values = list(range(rf_pca_coarse_n_estimators_values[best_n_index - 1], best_rf_pca_n_estimators + 101, 50))
else:
    refined_rf_pca_n_estimators_values = list(range(rf_pca_coarse_n_estimators_values[best_n_index - 1], rf_pca_coarse_n_estimators_values[best_n_index + 1] + 1, 50))

best_depth_index = rf_pca_coarse_max_depth_values.index(best_rf_pca_max_depth)

if best_depth_index == 0:
    lower_depth = 1
    upper_depth = rf_pca_coarse_max_depth_values[best_depth_index + 1]
elif best_depth_index == len(rf_pca_coarse_max_depth_values) - 1:
    lower_depth = rf_pca_coarse_max_depth_values[best_depth_index - 1]
    upper_depth = best_rf_pca_max_depth * 2
else:
    lower_depth = rf_pca_coarse_max_depth_values[best_depth_index - 1]
    upper_depth = rf_pca_coarse_max_depth_values[best_depth_index + 1]
depth_step = 8 if upper_depth <= 64 else 16

refined_rf_pca_max_depth_values = list(range(lower_depth, upper_depth + 1, depth_step))

random_forest_pca_refined_result_df = run_random_forest_pca_search(
    pca_component_values=refined_rf_pca_component_values,
    n_estimators_values=refined_rf_pca_n_estimators_values,
    max_depth_values=refined_rf_pca_max_depth_values,
    max_features_values=[best_rf_pca_max_features],
    stage_name="refined_search"
)

In [56]:
random_forest_pca_results_df = pd.concat([random_forest_pca_coarse_result_df, random_forest_pca_refined_result_df], ignore_index=True)
random_forest_pca_results_df = random_forest_pca_results_df.sort_values(by="validation_accuracy", ascending=False)

best_random_forest_pca = random_forest_pca_results_df.iloc[0]

best_rf_pca_components = int(best_random_forest_pca["pca_components"])
best_rf_pca_n_estimators = int(best_random_forest_pca["n_estimators"])
best_rf_pca_max_depth = int(best_random_forest_pca["max_depth"])
best_rf_pca_max_features = best_random_forest_pca["max_features"]

random_forest_pca_results_df.head(10)

,model,feature_type,search_stage,pca_components,preserved_variance,n_estimators,max_depth,max_features,validation_accuracy
72,Random Forest,PCA Vectors,refined_search,36,0.766890,300,32,log2,0.9536
73,Random Forest,PCA Vectors,refined_search,52,0.831187,250,32,log2,0.9535
78,Random Forest,PCA Vectors,refined_search,68,0.870348,300,96,log2,0.9534
79,Random Forest,PCA Vectors,refined_search,36,0.766890,250,32,log2,0.9534
75,Random Forest,PCA Vectors,refined_search,68,0.870348,300,80,log2,0.9534
74,Random Forest,PCA Vectors,refined_search,68,0.870348,300,128,log2,0.9534
76,Random Forest,PCA Vectors,refined_search,68,0.870348,300,64,log2,0.9534
77,Random Forest,PCA Vectors,refined_search,68,0.870348,300,112,log2,0.9534
80,Random Forest,PCA Vectors,refined_search,68,0.870348,300,48,log2,0.9534
82,Random Forest,PCA Vectors,refined_search,36,0.766890,250,48,log2,0.9531


In [57]:
validation_results.append({
    "model": "Random Forest",
    "feature_type": "PCA Vectors",
    "best_parameters": {
        "pca_components": best_rf_pca_components,
        "n_estimators": best_rf_pca_n_estimators,
        "max_depth": best_rf_pca_max_depth,
        "max_features": best_rf_pca_max_features
    },
    "validation_accuracy": best_random_forest_pca["validation_accuracy"]
})

pd.DataFrame(validation_results)

,model,feature_type,best_parameters,validation_accuracy
0,Decision Tree,Raw Vectors,"{'criterion': 'entropy', 'max_depth': 64, 'min...",0.8739
1,Random Forest,Raw Vectors,"{'n_estimators': 300, 'max_depth': 24, 'max_fe...",0.9697
2,Gradient Boosting,Raw Vectors,"{'learning_rate': 0.21, 'max_depth': 10, 'max_...",0.9756
3,Decision Tree,PCA Vectors,"{'pca_components': 20, 'criterion': 'entropy',...",0.8512
4,Random Forest,PCA Vectors,"{'pca_components': 36, 'n_estimators': 300, 'm...",0.9536


In [58]:
gb_fixed_max_iter  = 50

def run_gradient_boosting_pca_search(pca_component_values, learning_rate_values, max_depth_values, stage_name):
    results = []

    for component_count in pca_component_values:
        current_train_vectors = pca_model_train_all[: , : component_count]
        current_validation_vectors = pca_validation_all[:, : component_count]
        preserved_variance = pca_preserved_variance[component_count - 1]

        for learning_rate in learning_rate_values:
            for max_depth in max_depth_values:

                model = HistGradientBoostingClassifier(
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    max_iter=gb_fixed_max_iter,
                    random_state=42,
                    early_stopping=True,
                    n_iter_no_change=5,
                    validation_fraction=0.1,
                    max_bins=32
                )

                model.fit(current_train_vectors, model_train_labels)
                predictions = model.predict(current_validation_vectors)
                accuracy = calculate_accuracy(validation_labels, predictions)

                results.append({
                    "model": "Gradient Boosting",
                    "feature_type": "PCA Vectors",
                    "search_stage": stage_name,
                    "pca_components": component_count,
                    "preserved_variance": preserved_variance,
                    "learning_rate": learning_rate,
                    "max_depth": max_depth,
                    "max_iter": gb_fixed_max_iter,
                    "validation_accuracy": accuracy
                })

    return pd.DataFrame(results).sort_values(by="validation_accuracy", ascending=False)

In [59]:
gb_pca_coarse_learning_rate_values = [0.001, 0.01, 0.1, 0.3]
gb_pca_coarse_max_depth_values = [1, 2, 4, 8]

gradient_boosting_pca_coarse_result_df = run_gradient_boosting_pca_search(
    pca_component_values=pca_coarse_component_values,
    learning_rate_values=gb_pca_coarse_learning_rate_values,
    max_depth_values=gb_pca_coarse_max_depth_values,
    stage_name="coarse_search"
)

gradient_boosting_pca_coarse_result_df.head(10)

,model,feature_type,search_stage,pca_components,preserved_variance,learning_rate,max_depth,max_iter,validation_accuracy
43,Gradient Boosting,PCA Vectors,coarse_search,200,0.966518,0.1,8,50,0.9456
27,Gradient Boosting,PCA Vectors,coarse_search,100,0.914768,0.1,8,50,0.9456
11,Gradient Boosting,PCA Vectors,coarse_search,20,0.644021,0.1,8,50,0.9416
30,Gradient Boosting,PCA Vectors,coarse_search,100,0.914768,0.3,4,50,0.9362
46,Gradient Boosting,PCA Vectors,coarse_search,200,0.966518,0.3,4,50,0.9282
26,Gradient Boosting,PCA Vectors,coarse_search,100,0.914768,0.1,4,50,0.9246
14,Gradient Boosting,PCA Vectors,coarse_search,20,0.644021,0.3,4,50,0.9239
42,Gradient Boosting,PCA Vectors,coarse_search,200,0.966518,0.1,4,50,0.9233
15,Gradient Boosting,PCA Vectors,coarse_search,20,0.644021,0.3,8,50,0.9231
31,Gradient Boosting,PCA Vectors,coarse_search,100,0.914768,0.3,8,50,0.9190


In [60]:
best_gb_pca_coarse = gradient_boosting_pca_coarse_result_df.iloc[0]

best_gb_pca_component = int(best_gb_pca_coarse["pca_components"])
best_gb_pca_learning_rate = float(best_gb_pca_coarse["learning_rate"])
best_gb_pca_max_depth = int(best_gb_pca_coarse["max_depth"])

refined_gb_pca_component_values = create_refined_pca_values(best_gb_pca_component, pca_coarse_component_values)

best_lr_index = gb_pca_coarse_learning_rate_values.index(best_gb_pca_learning_rate)

if best_lr_index == 0:
    lower_lr = gb_pca_coarse_learning_rate_values[0]
    upper_lr = gb_pca_coarse_learning_rate_values[1]
elif best_lr_index == len(gb_pca_coarse_learning_rate_values) - 1:
    lower_lr = gb_pca_coarse_learning_rate_values[-2]
    upper_lr = gb_pca_coarse_learning_rate_values[-1] * 1.5
else:
    lower_lr = gb_pca_coarse_learning_rate_values[best_lr_index - 1]
    upper_lr = gb_pca_coarse_learning_rate_values[best_lr_index + 1]

refined_gb_pca_learning_rate_values = list(np.round(np.linspace(lower_lr, upper_lr, 7), 4))

best_depth_index = gb_pca_coarse_max_depth_values.index(best_gb_pca_max_depth)

if best_depth_index == 0:
    lower_depth = 1
    upper_depth = gb_pca_coarse_max_depth_values[best_depth_index + 1]
elif best_depth_index == len(gb_pca_coarse_max_depth_values) - 1:
    lower_depth = gb_pca_coarse_max_depth_values[best_depth_index - 1]
    upper_depth = best_gb_pca_max_depth * 2
else:
    lower_depth = gb_pca_coarse_max_depth_values[best_depth_index - 1]
    upper_depth = gb_pca_coarse_max_depth_values[best_depth_index + 1]

depth_step = 1 if upper_depth <= 8 else 2

refined_gb_pca_max_depth_values = list(range(lower_depth, upper_depth + 1, depth_step))

gradient_boosting_pca_refined_result_df = run_gradient_boosting_pca_search(
    pca_component_values=refined_gb_pca_component_values,
    learning_rate_values=refined_gb_pca_learning_rate_values,
    max_depth_values=refined_gb_pca_max_depth_values,
    stage_name="refined_search"
)

In [61]:
gradient_boosting_pca_results_df = pd.concat([gradient_boosting_pca_coarse_result_df, gradient_boosting_pca_refined_result_df], ignore_index=True)
gradient_boosting_pca_results_df = gradient_boosting_pca_results_df.sort_values(by="validation_accuracy", ascending=False)

best_gradient_boosting_pca = gradient_boosting_pca_results_df.iloc[0]

best_gb_pca_components = int(best_gradient_boosting_pca["pca_components"])
best_gb_pca_learning_rate = float(best_gradient_boosting_pca["learning_rate"])
best_gb_pca_max_depth = int(best_gradient_boosting_pca["max_depth"])

gradient_boosting_pca_results_df.head(10)

,model,feature_type,search_stage,pca_components,preserved_variance,learning_rate,max_depth,max_iter,validation_accuracy
48,Gradient Boosting,PCA Vectors,refined_search,132,0.939028,0.2033,12,50,0.9576
49,Gradient Boosting,PCA Vectors,refined_search,132,0.939028,0.2517,12,50,0.9574
50,Gradient Boosting,PCA Vectors,refined_search,132,0.939028,0.2033,8,50,0.9573
51,Gradient Boosting,PCA Vectors,refined_search,116,0.928383,0.2033,16,50,0.9573
52,Gradient Boosting,PCA Vectors,refined_search,100,0.914768,0.2033,16,50,0.9571
53,Gradient Boosting,PCA Vectors,refined_search,148,0.947526,0.2033,8,50,0.9571
54,Gradient Boosting,PCA Vectors,refined_search,180,0.960359,0.2033,14,50,0.9568
56,Gradient Boosting,PCA Vectors,refined_search,132,0.939028,0.2033,10,50,0.9568
55,Gradient Boosting,PCA Vectors,refined_search,116,0.928383,0.2033,10,50,0.9568
57,Gradient Boosting,PCA Vectors,refined_search,132,0.939028,0.2517,14,50,0.9567


In [62]:
validation_results.append({
    "model": "Gradient Boosting",
    "feature_type": "PCA Vectors",
    "best_parameters": {
        "pca_components": best_gb_pca_components,
        "learning_rate": best_gb_pca_learning_rate,
        "max_depth": best_gb_pca_max_depth,
        "max_iter": gb_fixed_max_iter
    },
    "validation_accuracy": best_gradient_boosting_pca["validation_accuracy"]
})

pd.DataFrame(validation_results)

,model,feature_type,best_parameters,validation_accuracy
0,Decision Tree,Raw Vectors,"{'criterion': 'entropy', 'max_depth': 64, 'min...",0.8739
1,Random Forest,Raw Vectors,"{'n_estimators': 300, 'max_depth': 24, 'max_fe...",0.9697
2,Gradient Boosting,Raw Vectors,"{'learning_rate': 0.21, 'max_depth': 10, 'max_...",0.9756
3,Decision Tree,PCA Vectors,"{'pca_components': 20, 'criterion': 'entropy',...",0.8512
4,Random Forest,PCA Vectors,"{'pca_components': 36, 'n_estimators': 300, 'm...",0.9536
5,Gradient Boosting,PCA Vectors,"{'pca_components': 132, 'learning_rate': 0.203...",0.9576


In [63]:
test_results = []

def evaluate_model(model_name, feature_type, model, train_vectors, test_vectors,best_parameters, validation_accuracy):
    model.fit(train_vectors, train_labels)
    predictions = model.predict(test_vectors)

    test_accuracy = calculate_accuracy(test_labels, predictions)
    confusion = calculate_confusion_matrix(test_labels, predictions)

    test_results.append({
        "model": model_name,
        "feature_type": feature_type,
        "best_parameters": best_parameters,
        "validation_accuracy": validation_accuracy,
        "test_accuracy": test_accuracy,
        "confusion_matrix": confusion
    })


def evaluate_pca_model(model_name, model, component_count, best_parameters, validation_accuracy):
    final_pca_model = PCA(n_components=component_count, svd_solver="randomized", random_state=42)

    final_train_vectors = final_pca_model.fit_transform(normalized_train_vectors)
    final_test_vectors = final_pca_model.transform(normalized_test_vectors)

    final_parameters = best_parameters.copy()
    final_parameters["pca_components"] = component_count
    final_parameters["preserved_variance"] = np.sum(final_pca_model.explained_variance_ratio_)

    evaluate_model(
        model_name,
        "PCA Vectors",
        model,
        final_train_vectors,
        final_test_vectors,
        final_parameters,
        validation_accuracy
    )

In [64]:
evaluate_model(
    "Decision Tree",
    "Raw Vectors",
    DecisionTreeClassifier(
        criterion=best_dt_raw_criterion,
        max_depth=best_dt_raw_max_depth,
        min_samples_split=best_dt_raw_min_samples_split,
        random_state=42
    ),
    normalized_train_vectors,
    normalized_test_vectors,
    {
        "criterion": best_dt_raw_criterion,
        "max_depth": best_dt_raw_max_depth,
        "min_samples_split": best_dt_raw_min_samples_split
    },
    best_decision_tree_raw["validation_accuracy"]
)

evaluate_model(
    "Random Forest",
    "Raw Vectors",
    RandomForestClassifier(
        n_estimators=best_rf_raw_n_estimators,
        max_depth=best_rf_raw_max_depth,
        max_features=best_rf_raw_max_features,
        random_state=42,
        n_jobs=-1
    ),
    normalized_train_vectors,
    normalized_test_vectors,
    {
        "n_estimators": best_rf_raw_n_estimators,
        "max_depth": best_rf_raw_max_depth,
        "max_features": best_rf_raw_max_features
    },
    best_random_forest_raw["validation_accuracy"]
)

evaluate_model(
    "Gradient Boosting",
    "Raw Vectors",
    HistGradientBoostingClassifier(
        learning_rate=best_gb_raw_learning_rate,
        max_depth=best_gb_raw_max_depth,
        max_iter=gb_fixed_max_iter,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=5,
        validation_fraction=0.1,
        max_bins=32
    ),
    normalized_train_vectors,
    normalized_test_vectors,
    {
        "learning_rate": best_gb_raw_learning_rate,
        "max_depth": best_gb_raw_max_depth,
        "max_iter": gb_fixed_max_iter
    },
    best_gradient_boosting_raw["validation_accuracy"]
)

In [65]:
evaluate_pca_model(
    "Decision Tree",
    DecisionTreeClassifier(
        criterion=best_dt_pca_criterion,
        max_depth=best_dt_pca_max_depth,
        min_samples_split=best_dt_pca_min_samples_split,
        random_state=42
    ),
    best_dt_pca_components,
    {
        "criterion": best_dt_pca_criterion,
        "max_depth": best_dt_pca_max_depth,
        "min_samples_split": best_dt_pca_min_samples_split
    },
    best_decision_tree_pca["validation_accuracy"]
)

evaluate_pca_model(
    "Random Forest",
    RandomForestClassifier(
        n_estimators=best_rf_pca_n_estimators,
        max_depth=best_rf_pca_max_depth,
        max_features=best_rf_pca_max_features,
        random_state=42,
        n_jobs=-1
    ),
    best_rf_pca_components,
    {
        "n_estimators": best_rf_pca_n_estimators,
        "max_depth": best_rf_pca_max_depth,
        "max_features": best_rf_pca_max_features
    },
    best_random_forest_pca["validation_accuracy"]
)

evaluate_pca_model(
    "Gradient Boosting",
    HistGradientBoostingClassifier(
        learning_rate=best_gb_pca_learning_rate,
        max_depth=best_gb_pca_max_depth,
        max_iter=gb_fixed_max_iter,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=5,
        validation_fraction=0.1,
        max_bins=32
    ),
    best_gb_pca_components,
    {
        "learning_rate": best_gb_pca_learning_rate,
        "max_depth": best_gb_pca_max_depth,
        "max_iter": gb_fixed_max_iter
    },
    best_gradient_boosting_pca["validation_accuracy"]
)

In [66]:
test_results.append({
    "model": "KNN",
    "feature_type": "PCA Vectors",
    "best_parameters": {
        "pca_components": 50,
        "k": 3
    },
    "validation_accuracy": 0.9784,
    "test_accuracy": 0.9752,
    "confusion_matrix": "Reported in Assignment 1"
})

final_comparison_table = pd.DataFrame([
    {
        "Model": result["model"],
        "Feature Type": result["feature_type"],
        "Best Parameters": result["best_parameters"],
        "Validation Accuracy": result["validation_accuracy"],
        "Test Accuracy": result["test_accuracy"]
    }
    for result in test_results
])

final_comparison_table = final_comparison_table.sort_values(
    by="Test Accuracy",
    ascending=False
)

final_comparison_table

,Model,Feature Type,Best Parameters,Validation Accuracy,Test Accuracy
2,Gradient Boosting,Raw Vectors,"{'learning_rate': 0.21, 'max_depth': 10, 'max_...",0.9756,0.9755
6,KNN,PCA Vectors,"{'pca_components': 50, 'k': 3}",0.9784,0.9752
1,Random Forest,Raw Vectors,"{'n_estimators': 300, 'max_depth': 24, 'max_fe...",0.9697,0.9716
4,Random Forest,PCA Vectors,"{'n_estimators': 300, 'max_depth': 32, 'max_fe...",0.9536,0.9575
5,Gradient Boosting,PCA Vectors,"{'learning_rate': 0.2033, 'max_depth': 12, 'ma...",0.9576,0.9416
0,Decision Tree,Raw Vectors,"{'criterion': 'entropy', 'max_depth': 64, 'min...",0.8739,0.8857
3,Decision Tree,PCA Vectors,"{'criterion': 'entropy', 'max_depth': 16, 'min...",0.8512,0.8603


In [67]:
for result in test_results:
    print(result["model"], "-", result["feature_type"])
    print("Test accuracy:", result["test_accuracy"])
    print("Confusion matrix:")
    print(result["confusion_matrix"])
    print()

Decision Tree - Raw Vectors
Test accuracy: 0.8857
Confusion matrix:
[[ 910    1   10    5    4   18   16    4    6    6]
 [   0 1096    9    3    2    7    3    2   10    3]
 [   7    5  911   21   11    9   19   16   28    5]
 [   6    8   28  852    8   43    4   14   27   20]
 [   9    2   15    5  866    6   15    9   17   38]
 [  15    4    7   37    7  751   20    4   22   25]
 [  16    4   10    7   23   19  863    4   10    2]
 [   2    9   33   16   13    5    3  924    8   15]
 [   6    6   21   30   19   23   16   10  816   27]
 [   8    3    7   20   35   18    6   16   28  868]]

Random Forest - Raw Vectors
Test accuracy: 0.9716
Confusion matrix:
[[ 970    0    0    0    0    2    2    1    4    1]
 [   0 1123    3    2    0    2    3    0    1    1]
 [   6    0 1001    4    3    0    4    8    6    0]
 [   0    0    9  974    0    6    0    9    9    3]
 [   1    0    2    0  954    0    4    0    2   19]
 [   3    0    0    8    3  863    6    2    5    2]
 [   7    3   